# YETI CSV cleaning for the knowledge graph

Prepare a selected YETI CSV for Neo4j import. Choose the input CSV in the configuration cell below, then run the notebook from the top. The source file is left unchanged; the cleaned output is written to `data/processed/<input-name>_shipments_clean.csv`.

The output is a denormalized staging table: each row represents one shipment and includes IDs for the shipment and its buyer, supplier, ports, product, carrier, and vessel. It retains source names and shipment-level context so graph nodes and relationships can be created from the same file. Organization IDs use the source's legal company names to avoid merging distinct legal entities that happen to share a basename. Basenames remain available as grouping clues for later entity resolution.

Records without a bill-of-lading number are retained and receive an ID based on their row in this source-file snapshot. If the source file is reordered or refreshed, review those fallback IDs before merging them into an existing graph.

In [ ]:
import re
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data/csv").is_dir()
)
CSV_DIR = PROJECT_ROOT / "data/csv"
AVAILABLE_CSVS = sorted(path.name for path in CSV_DIR.glob("*.csv"))

# Choose any YETI CSV in data/csv, then run the notebook from the top.
SOURCE_CSV = "flextronics.csv"

if SOURCE_CSV not in AVAILABLE_CSVS:
    raise FileNotFoundError(
        f"{SOURCE_CSV!r} was not found in {CSV_DIR}. Available CSVs: {AVAILABLE_CSVS}"
    )

SOURCE_PATH = CSV_DIR / SOURCE_CSV
OUTPUT_PATH = PROJECT_ROOT / "data/processed" / f"{SOURCE_PATH.stem}_shipments_clean.csv"

raw = pd.read_csv(SOURCE_PATH, dtype="string", keep_default_na=False)
raw.insert(0, "source_row", range(1, len(raw) + 1))
print(f"Loaded {len(raw):,} rows and {len(raw.columns) - 1:,} source columns from {SOURCE_PATH.name}.")
display(raw.head(3))

## Select and standardize the target records

The selected CSV is cleaned as a whole: no supplier-specific row filter is applied, so shipments involving any company in that extract are retained. Text is trimmed without changing display spelling or merging similarly named legal entities.

In [ ]:
def clean_text(series: pd.Series) -> pd.Series:
    values = series.astype("string").str.strip()
    return values.mask(values.eq(""), pd.NA)


def make_key(prefix: str, value: object) -> object:
    if pd.isna(value):
        return pd.NA
    slug = re.sub(r"[^a-z0-9]+", "-", str(value).casefold()).strip("-")
    return f"{prefix}:{slug}" if slug else pd.NA


text_columns = raw.select_dtypes(include="string").columns
source = raw.copy()
source[text_columns] = source[text_columns].apply(clean_text)

shipments = source.copy()
if shipments.empty:
    raise ValueError(f"The selected CSV contains no shipment rows: {SOURCE_PATH}")

print(f"Selected {len(shipments):,} rows from {SOURCE_PATH.name}; all source rows are retained.")
display(shipments[["supplier_name", "supplier_basename", "supplier_country"]].value_counts(dropna=False).rename("rows").reset_index())

## Build the graph-ready shipment table

IDs are prefixed by entity type and use legal company names for conservative buyer/supplier matching. Port, HS product, carrier, and vessel IDs use the most specific source identifiers available, falling back to a readable name where needed. Dates are written in ISO `YYYY-MM-DD` form; numeric fields are parsed as numbers and invalid/missing values remain empty.

In [ ]:
clean = pd.DataFrame(index=shipments.index)
clean["source_file"] = SOURCE_PATH.name
clean["source_row"] = shipments["source_row"].astype("Int64")

clean["bol_number"] = clean_text(shipments["bol_number"]).str.upper()
clean["shipment_id"] = clean["bol_number"].map(lambda value: make_key("bol", value))
missing_bol = clean["shipment_id"].isna()
clean.loc[missing_bol, "shipment_id"] = clean.loc[missing_bol, "source_row"].map(
    lambda row: f"{SOURCE_PATH.stem}:row:{int(row):05d}"
)

field_map = {
    "bol_type": "bol_type",
    "master_bol_number": "master_bol_number",
    "has_master_data": "has_master_data",
    "buyer_name": "company_name",
    "buyer_basename": "company_basename",
    "buyer_address": "company_address",
    "buyer_country": "company_country",
    "buyer_country_code": "company_country_code",
    "buyer_first_shipment_date": "company_first_shipment_date",
    "buyer_total_shipments": "company_total_shipments",
    "supplier_name": "supplier_name",
    "supplier_basename": "supplier_basename",
    "supplier_address": "supplier_address",
    "supplier_country": "supplier_country",
    "supplier_country_code": "supplier_country_code",
    "supplier_address_country_code": "supplier_address_country_code",
    "supplier_first_shipment_date": "supplier_first_shipment_date",
    "supplier_total_shipments": "supplier_total_shipments",
    "entry_port": "entry_port",
    "entry_port_code": "entry_port_code",
    "exit_port": "exit_port",
    "exit_port_code": "exit_port_code",
    "hs_code": "hs_code",
    "hs_code_description": "hs_code_description",
    "full_hs_code_description": "full_hs_code_description",
    "product_description": "product_description",
    "carrier_name": "carrier_scac_code",
    "vessel_code": "vessel_code",
    "vessel_name": "vessel_name",
    "vessel_country_code": "vessel_country_code",
    "voyage": "voyage",
    "place_of_receipt": "place_of_receipt",
    "quantity_unit": "quantity_unit",
    "container_types": "container_types",
    "internal_supplier": "internal_supplier",
    "uflpa_buyer": "uflpa_company",
    "uflpa_supplier": "uflpa_supplier",
    "buyer_china_concentration": "company_china_concentration",
}
for output_name, source_name in field_map.items():
    clean[output_name] = clean_text(shipments[source_name])

for output_name, source_name in {
    "arrival_date": "arrival_date",
    "estimated_arrival_date": "estimated_arrival_date",
    "buyer_first_shipment_date": "company_first_shipment_date",
    "supplier_first_shipment_date": "supplier_first_shipment_date",
}.items():
    date_format = "%Y%m%d" if output_name == "estimated_arrival_date" else "%m/%d/%Y"
    parsed = pd.to_datetime(shipments[source_name], format=date_format, errors="coerce")
    clean[output_name] = parsed.dt.strftime("%Y-%m-%d").astype("string")

numeric_fields = {
    "quantity": "quantity",
    "weight": "weight",
    "teu": "teu",
    "containers_count": "containers_count",
    "shipping_cost": "shipping_cost",
    "cif": "cif",
    "buyer_total_shipments": "company_total_shipments",
    "supplier_total_shipments": "supplier_total_shipments",
    "buyer_china_concentration": "company_china_concentration",
    "entry_port_lat": "entry_port_geocode.lat",
    "entry_port_lon": "entry_port_geocode.lon",
    "exit_port_lat": "exit_port_geocode.lat",
    "exit_port_lon": "exit_port_geocode.lon",
}
for output_name, source_name in numeric_fields.items():
    clean[output_name] = pd.to_numeric(shipments[source_name], errors="coerce")

for flag in ("has_master_data", "internal_supplier", "uflpa_buyer", "uflpa_supplier"):
    clean[flag] = clean[flag].str.casefold().map({"true": True, "false": False}).astype("boolean")

clean["buyer_id"] = clean["buyer_name"].map(lambda value: make_key("org", value))
clean["supplier_id"] = clean["supplier_name"].map(lambda value: make_key("org", value))
clean["entry_port_id"] = [
    make_key("port", code) if pd.notna(code) else make_key("port", name)
    for code, name in zip(clean["entry_port_code"], clean["entry_port"], strict=True)
]
clean["exit_port_id"] = [
    make_key("port", code) if pd.notna(code) else make_key("port", name)
    for code, name in zip(clean["exit_port_code"], clean["exit_port"], strict=True)
]
clean["product_id"] = [
    make_key("hs", hs_code) if pd.notna(hs_code) else make_key("product", description)
    for hs_code, description in zip(clean["hs_code"], clean["product_description"], strict=True)
]
clean["carrier_code"] = clean["carrier_name"].str.split(" - ", n=1).str[0].str.strip()
clean["carrier_id"] = [
    make_key("carrier", code) if pd.notna(code) else make_key("carrier", name)
    for code, name in zip(clean["carrier_code"], clean["carrier_name"], strict=True)
]
clean["vessel_id"] = [
    make_key("vessel", code) if pd.notna(code) else make_key("vessel", name)
    for code, name in zip(clean["vessel_code"], clean["vessel_name"], strict=True)
]

graph_columns = [
    "source_file", "source_row", "shipment_id", "bol_number", "bol_type", "master_bol_number", "has_master_data",
    "arrival_date", "estimated_arrival_date", "buyer_first_shipment_date", "supplier_first_shipment_date",
    "buyer_id", "buyer_name", "buyer_basename", "buyer_address", "buyer_country", "buyer_country_code", "buyer_total_shipments",
    "supplier_id", "supplier_name", "supplier_basename", "supplier_address", "supplier_country", "supplier_country_code",
    "supplier_address_country_code", "supplier_total_shipments",
    "entry_port_id", "entry_port", "entry_port_code", "entry_port_lat", "entry_port_lon",
    "exit_port_id", "exit_port", "exit_port_code", "exit_port_lat", "exit_port_lon",
    "product_id", "hs_code", "hs_code_description", "full_hs_code_description", "product_description",
    "quantity", "quantity_unit", "weight", "teu", "containers_count", "container_types",
    "carrier_id", "carrier_code", "carrier_name", "vessel_id", "vessel_code", "vessel_name", "vessel_country_code", "voyage", "place_of_receipt",
    "shipping_cost", "cif", "internal_supplier", "uflpa_buyer", "uflpa_supplier", "buyer_china_concentration",
]
clean = clean[graph_columns].reset_index(drop=True)
display(clean.head(5))
print(f"Prepared {len(clean):,} shipment rows and {len(clean.columns):,} graph-relevant columns.")

## Validate and export

Do not silently discard rows or duplicate known bills of lading. The checks below fail visibly if the shipment IDs are not unique or required graph-link keys are missing. Empty source values remain empty in the CSV rather than being replaced with misleading defaults.

In [ ]:
known_bols = clean["bol_number"].dropna()
assert not known_bols.duplicated().any(), "A non-empty BOL number occurs more than once; review before graph import."
assert clean["shipment_id"].notna().all(), "Every shipment must have an ID."
assert clean["shipment_id"].is_unique, "Shipment IDs must be unique."
for required_id in ("buyer_id", "supplier_id", "entry_port_id", "exit_port_id", "product_id"):
    assert clean[required_id].notna().all(), f"Missing graph key in {required_id}."

date_quality = pd.DataFrame({
    "field": ["arrival_date", "estimated_arrival_date", "buyer_first_shipment_date", "supplier_first_shipment_date"],
    "missing_or_unparsed": [int(clean[column].isna().sum()) for column in (
        "arrival_date", "estimated_arrival_date", "buyer_first_shipment_date", "supplier_first_shipment_date"
    )],
})
print(f"Rows retained: {len(clean):,}")
print(f"Rows with BOL number: {len(known_bols):,}; fallback IDs used: {int(clean['bol_number'].isna().sum()):,}")
print(f"Distinct buyers: {clean['buyer_id'].nunique()}; suppliers: {clean['supplier_id'].nunique()}; products: {clean['product_id'].nunique()}")
display(date_quality)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
round_trip = pd.read_csv(OUTPUT_PATH, dtype={"shipment_id": "string", "bol_number": "string"})
assert len(round_trip) == len(clean), "Export row count did not match the cleaned table."
assert round_trip["shipment_id"].is_unique, "Exported shipment IDs are not unique."
print(f"Wrote and verified {len(round_trip):,} rows to {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")

## Neo4j loading shape

Use `shipment_id` as the unique key for `Shipment` nodes. `buyer_id`, `supplier_id`, `entry_port_id`, `exit_port_id`, `product_id`, `carrier_id`, and `vessel_id` are the corresponding `MERGE` keys. Each row can create `(:Organization)-[:SENT]->(:Shipment)-[:DELIVERED_TO]->(:Organization)`, plus `DEPARTED_FROM`, `ARRIVED_AT`, `CLASSIFIED_AS`, `MOVED_BY`, and `ON_VESSEL` relationships. Keep the source basenames as reconciliation clues; only merge entities on a basename after confirming they represent the same organization.